# Le Xin — timing correction starter

Hi Le Xin! This is your starting point, not a finished model. We load the handover tables, check labels, and measure what happens when we change nothing. Your model choice and training are still yours to do.

**Labels are PROVISIONAL pending group manual review.** Please review a few examples with the group before treating the targets as ground truth. This notebook does not fit a model and never opens test data.

### Keep the boundary clear
- `X` contains only prediction-derived allowlisted features. Reference matches, labels, IDs and split metadata are not model inputs.
- Note features: `start, end, pitch, amplitude, duration, prev_gap, next_gap`; pair features: `gap, pitch_delta, duration_sum`.
- Missing/ambiguous labels are excluded only from this task's training-label diagnostics, never from full end-to-end evaluation.
- Timing targets are absolute seconds; pitch targets are absolute MIDI. Positive classifier labels mean merge the pair / remove the false note.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# Open from this notebook folder, or run from the project root.
HERE = next(p / 'notebooks' / 'correction_handover' for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'GTSinger_100_unique_sample'] if (p / 'selected_100.csv').exists())
sys.path.insert(0, str(HERE))
import correction_utils as cu
DATA = HERE / '..' / '..' / 'basic_pitch_runs' / 'correction_handover'
notes = pd.read_csv(DATA / 'train_features.csv', dtype={'note_id':str,'recording_id':str,'sample_id':str})
note_targets = pd.read_csv(DATA / 'proposed_v2' / 'train_targets_v2.csv', dtype={'note_id':str}).rename(columns={'false_note':'false_label','reference_start':'target_start','reference_end':'target_end','reference_pitch':'target_pitch'})
assert notes['split'].eq('train').all(), 'Only train data belongs in this starter.'
manifest = pd.read_csv(DATA / 'training_recording_manifest.csv')
cu.validate_notes(notes, manifest)
features, targets = notes, note_targets
TASK = 'timing'
X, joined = cu.training_data(features, targets, TASK)
print('Feature allowlist:', X.columns.tolist())
print('Rows:', len(X), '| target-row coverage: 100%')
display(X.head())


Feature allowlist: ['start', 'end', 'pitch', 'amplitude', 'duration', 'prev_gap', 'next_gap']
Rows: 7623 | target-row coverage: 100%


,start,end,pitch,amplitude,duration,prev_gap,next_gap
0,0.394739,0.684989,55.0,0.729802,0.290249,0.00000,0.092880
1,0.777868,0.986848,49.0,0.616202,0.208980,0.09288,0.034830
2,1.021678,1.265488,54.0,0.627965,0.243810,0.03483,0.150930
3,1.416417,1.625397,49.0,0.576493,0.208980,0.15093,0.092880
4,1.718277,1.915646,55.0,0.518624,0.197370,0.09288,0.082554


## Check what we can actually learn from

Target rows must cover every feature ID, even when their task labels are missing. For classification, inspect both class counts before choosing a method. For timing/pitch, the value counts below describe target coverage rather than classifier classes.

In [2]:
label_columns = cu.TARGETS[TASK]
usable = joined[label_columns].notna().all(axis=1)
if 'ambiguous' in joined:
    # CSV accepts True/False or 1/0. Reject unknown encodings rather than guessing.
    ambiguity = joined['ambiguous'].astype(str).str.lower().map({'true':True,'false':False,'1':True,'0':False})
    assert ambiguity.notna().all(), 'ambiguous must be explicit True/False or 1/0'
    usable &= ~ambiguity
print({'total_rows':len(joined), 'usable_task_labels':int(usable.sum()),
       'excluded_task_labels':int((~usable).sum())})
for column in label_columns:
    print(column, 'non-null coverage:', float(joined[column].notna().mean()))
    display(joined.loc[usable,column].value_counts(dropna=False).rename('count'))
labelled = joined.loc[usable].copy()
X_task = X.loc[usable].copy()  # Only these allowlisted columns may reach your future model.
folds = pd.read_csv(DATA / 'folds.csv').set_index('sample_id')['fold']
groups = labelled['sample_id'].copy()
fold_ids = groups.map(folds)
assert fold_ids.notna().all(), 'Missing training-song fold'
print('Labelled examples per song-grouped fold:', fold_ids.value_counts().sort_index().to_dict())
print('Do not fit until the group approves the provisional label version.')


{'total_rows': 7623, 'usable_task_labels': 3697, 'excluded_task_labels': 3926}
target_start non-null coverage: 0.4849796667978486


target_start
0.000    180
0.240      6
2.450      5
0.850      5
2.840      5
        ... 
5.414      1
6.154      1
6.444      1
1.939      1
5.626      1
Name: count, Length: 2760, dtype: int64

target_end non-null coverage: 0.4849796667978486


target_end
0.850    7
1.250    6
0.570    6
4.891    6
2.540    6
        ..
6.444    1
6.754    1
1.296    1
5.626    1
5.768    1
Name: count, Length: 2872, dtype: int64

Labelled examples per song-grouped fold: {0: 678, 1: 856, 2: 827, 3: 671, 4: 665}
Do not fit until the group approves the provisional label version.


## Our first comparison: do nothing

This is a real no-change baseline on usable training task labels, not evidence of validation performance. Empty-label metrics have no useful interpretation. Nothing is fitted here.

In [3]:
if len(labelled) == 0:
    baseline = {'status':'BLOCKED_NO_REVIEWED_OR_PROVISIONAL_TARGETS', 'n':0, 'metrics':None}
else:
    baseline = cu.timing_metrics(labelled[['target_start','target_end']].to_numpy(), labelled[['start','end']].to_numpy())
print('No-change task baseline (not a trained model):', baseline)


No-change task baseline (not a trained model): {'n': 3697, 'onset_mae_seconds': 0.06883780659319899, 'onset_rmse_seconds': 0.0880691165106575, 'onset_median_absolute_error_ms': 51.040362811791255, 'offset_mae_seconds': 0.09175885074433707, 'offset_rmse_seconds': 0.1654258157667273, 'offset_median_absolute_error_ms': 42.477097505668304, 'onset_within_50ms': 0.4925615363808493, 'offset_within_tolerance': 0.720584257506086}


## Your model goes here — later

Use `X_task` and the separate labels when you are ready. Keep validation separate by song (`sample_id`), choose settings on train/val only, and leave test untouched. Do not infer features from reference alignment. For merge predictions, resolve overlapping pair decisions before producing a complete note table.

In [4]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# 1. What the model must learn: how far to move each boundary, in seconds.
#    (positive = the reference is later than Basic Pitch, so move the note later)
y_start = joined['target_start'] - joined['start']
y_end   = joined['target_end']   - joined['end']

# 2. Which song-fold each note belongs to (all notes, not just labelled ones)
fold_all = joined['sample_id'].map(folds)
assert fold_all.notna().all()

# 3. Train on 4 folds, predict the 5th. Repeat so every note gets a prediction
#    from a model that never saw its song.
def out_of_fold(make_model):
    pred = np.zeros((len(X), 2))            # column 0 = start shift, column 1 = end shift
    for k in sorted(fold_all.unique()):
        test  = (fold_all == k).to_numpy()
        train = (~test) & usable.to_numpy() # labelled notes from the other songs only
        for col, y in enumerate([y_start, y_end]):
            model = make_model().fit(X[train], y[train])
            pred[test, col] = model.predict(X[test])
    return pred

shift_dt = out_of_fold(lambda: DecisionTreeRegressor(min_samples_leaf=40, random_state=0))
shift_rf = out_of_fold(lambda: RandomForestRegressor(n_estimators=150, min_samples_leaf=20,
                                                     max_features=0.6, n_jobs=-1, random_state=0))

for name, s in [('decision tree', shift_dt), ('random forest', shift_rf)]:
    print(name, '| mean predicted start shift (ms):', round(s[:,0].mean()*1000,1),
          '| mean predicted end shift (ms):', round(s[:,1].mean()*1000,1))

decision tree | mean predicted start shift (ms): -45.2 | mean predicted end shift (ms): 60.5
random forest | mean predicted start shift (ms): -44.9 | mean predicted end shift (ms): 61.5


## Try the shared output interface

The CSV below contains the real original notes unchanged. A future correction CSV is the **complete replacement note set**, not a patch: omitted notes are deletions; for a merge retain one ID and omit absorbed IDs. Never overwrite the source tables.

In [5]:
# Real, unchanged notes: do not filter this with the task-label mask.
unchanged = notes[cu.NOTE_COLUMNS].copy(deep=True)
output = HERE / 'exports' / 'le_xin_timing_unchanged.csv'
if output.exists():
    # Safe notebook rerun: verify the existing export instead of overwriting it.
    existing = pd.read_csv(output, dtype={'note_id':str,'recording_id':str})
    pd.testing.assert_frame_equal(cu.validate_notes(existing, manifest), cu.validate_notes(unchanged, manifest), check_dtype=False)
else:
    cu.export_corrections(unchanged, manifest, output)
print('Unchanged-note interface example:', output)


Unchanged-note interface example: c:\Users\derp2\SMU\Y2SEM1\DMBA\GTSinger_Correction_Group_Handover\GTSinger_100_unique_sample\notebooks\correction_handover\exports\le_xin_timing_unchanged.csv


## Step 3: score the corrections

Two scores for each version of the notes:
- **Task metrics** (labelled notes only): how close the starts and ends are to the provisional labels.
- **End-to-end F1** (every training recording, full references): onset F1 and offset F1, the project's real measure.

The shifts come from the out-of-fold predictions in cell 7, so no song is scored by a model that was trained on it.

In [6]:
# Step 3: score the corrections, on labelled notes AND on every training recording

# 0. The end-to-end scorer needs librosa + mir_eval. If this kernel lacks them,
#    load le_xin_work/shim.py, which stands in for both (matches the published baseline within ~0.0003).
try:
    import librosa, mir_eval.transcription
    print('Scorer: real librosa + mir_eval')
except ImportError:
    sys.path.insert(0, str((HERE / '..' / '..' / 'le_xin_work').resolve()))
    import shim
    print('Scorer: shim.py stand-in (librosa/mir_eval not installed in this kernel)')

full_manifest   = pd.read_csv(DATA / 'training_recording_manifest.csv')
full_references = pd.read_csv(DATA / 'references_train.csv', dtype={'note_id':str,'recording_id':str})
audio_end = joined['recording_id'].map(full_manifest.set_index('recording_id')['duration_seconds_if_validated']).to_numpy()

# 1. Turn predicted shifts into a COMPLETE corrected note table (all 7,623 notes, same IDs).
#    Optional gates: ignore small shifts (|shift| < gate), since tiny moves are mostly noise.
#    base/end_limit default to the training notes; Step 6 passes the validation notes instead.
def apply_shifts(shift, gate_start=0.0, gate_end=0.0, base=None, end_limit=None):
    base = joined if base is None else base
    end_limit = audio_end if end_limit is None else end_limit
    s = shift.copy()
    s[np.abs(s[:, 0]) < gate_start, 0] = 0
    s[np.abs(s[:, 1]) < gate_end,   1] = 0
    out = base[cu.NOTE_COLUMNS].copy()
    out['start'] = np.maximum(0, base['start'] + s[:, 0])           # no negative times
    out['end']   = np.minimum(base['end'] + s[:, 1], end_limit)     # not past the end of the audio
    out['end']   = np.maximum(out['end'], out['start'] + 0.01)      # keep every note at least 10 ms long
    out['start'] = np.minimum(out['start'], out['end'] - 0.01)
    return out

# 2. Score one corrected table both ways.
truth = labelled[['target_start', 'target_end']].to_numpy()
def score(name, corrected):
    task = cu.timing_metrics(truth, corrected.loc[usable, ['start', 'end']].to_numpy())  # labelled notes only
    e2e  = cu.evaluate_end_to_end(corrected, full_references, full_manifest)['summary']  # every recording
    assert e2e['recordings'] == len(full_manifest)
    return {'model': name,
            'onset_mae_ms':   round(task['onset_mae_seconds'] * 1000, 1),
            'onset_within_50ms': round(task['onset_within_50ms'], 3),
            'offset_mae_ms':  round(task['offset_mae_seconds'] * 1000, 1),
            'offset_within_tol': round(task['offset_within_tolerance'], 3),
            'onset_F1':  round(e2e['song_macro_f1'], 4),
            'offset_F1': round(e2e['song_macro_offset_f1'], 4)}

results = pd.DataFrame([
    score('Basic Pitch unchanged', unchanged),
    score('Decision tree',         apply_shifts(shift_dt)),
    score('Random forest',         apply_shifts(shift_rf)),
    score('RF, gated 30/80 ms *',  apply_shifts(shift_rf, gate_start=0.03, gate_end=0.08)),
]).set_index('model')
display(results)
print('* gate picked by looking at these same out-of-fold scores, so this row is slightly optimistic.')

Scorer: real librosa + mir_eval


,onset_mae_ms,onset_within_50ms,offset_mae_ms,offset_within_tol,onset_F1,offset_F1
model,,,,,,
Basic Pitch unchanged,68.8,0.493,91.8,0.721,0.2628,0.1539
Decision tree,56.4,0.531,99.5,0.651,0.2899,0.1573
Random forest,54.9,0.531,93.0,0.680,0.2882,0.1636
"RF, gated 30/80 ms *",54.9,0.539,92.5,0.705,0.2923,0.1742


* gate picked by looking at these same out-of-fold scores, so this row is slightly optimistic.


## Step 4: export the corrected notes and the action history

The README asks for two files per correction:
- a **complete** corrected note table (`note_id, recording_id, start, end, pitch`), which replaces the original list (omitted notes would count as deleted);
- a separate **action history** listing what changed, for audit.

We export the plain random forest, with no gate, because no setting in it was chosen by looking at its own scores. Files are written once and never overwritten. On a rerun the cell checks that the saved files still match.

In [7]:
# Step 4: export the corrected note table + a separate action history

LABEL_VERSION = 'proposed_v2/train_targets_v2.csv (provisional)'
EXPORTS = HERE / 'exports'

def save_once(table, path, validate=False):
    """Write a CSV the first time; on a rerun, check the saved file is identical instead of overwriting it."""
    if path.exists():
        saved = pd.read_csv(path, dtype={'note_id':str, 'recording_id':str})
        pd.testing.assert_frame_equal(saved, table.reset_index(drop=True), check_dtype=False)
        print('Already saved, unchanged:', path.name)
    elif validate:
        cu.export_corrections(table, manifest, path)   # validates the complete note table, never overwrites
        print('Saved:', path.name)
    else:
        with path.open('x', encoding='utf-8', newline='') as handle:
            table.to_csv(handle, index=False)
        print('Saved:', path.name)

def export_version(name, shift, model_label, **gates):
    corrected = apply_shifts(shift, **gates)
    assert len(corrected) == len(notes) and corrected['note_id'].tolist() == notes['note_id'].tolist()
    assert corrected['pitch'].equals(notes['pitch'])                  # timing task: pitch untouched

    # Action history: one row per note whose start or end moved (nothing is deleted or merged here).
    old = notes.set_index('note_id')[['start', 'end']]
    new = corrected.set_index('note_id')[['start', 'end']]
    moved = (new['start'] - old['start']).abs().gt(1e-9) | (new['end'] - old['end']).abs().gt(1e-9)
    history = pd.DataFrame({
        'note_id':      new.index[moved],
        'recording_id': corrected.set_index('note_id').loc[moved, 'recording_id'].to_numpy(),
        'action':       'timing_change',
        'old_start':    old.loc[moved, 'start'].to_numpy(),
        'new_start':    new.loc[moved, 'start'].to_numpy(),
        'old_end':      old.loc[moved, 'end'].to_numpy(),
        'new_end':      new.loc[moved, 'end'].to_numpy(),
    })
    history['start_shift_ms'] = ((history['new_start'] - history['old_start']) * 1000).round(1)
    history['end_shift_ms']   = ((history['new_end']   - history['old_end'])   * 1000).round(1)
    history['model'] = model_label
    history['label_version'] = LABEL_VERSION

    save_once(corrected[cu.NOTE_COLUMNS], EXPORTS / f'le_xin_timing_{name}.csv', validate=True)
    save_once(history, EXPORTS / f'le_xin_timing_{name}_actions.csv')
    print(f'  {len(corrected)} notes kept, 0 deleted, {len(history)} timing changes '
          f'({history.start_shift_ms.ne(0).sum()} starts, {history.end_shift_ms.ne(0).sum()} ends moved)')
    return corrected, history

corrected_rf, history_rf = export_version(
    'rf_v2', shift_rf,
    'RandomForestRegressor(n_estimators=150, min_samples_leaf=20, max_features=0.6), out-of-fold by song')
display(history_rf.head())

Already saved, unchanged: le_xin_timing_rf_v2.csv
Already saved, unchanged: le_xin_timing_rf_v2_actions.csv
  7623 notes kept, 0 deleted, 7623 timing changes (7619 starts, 7622 ends moved)


,note_id,recording_id,action,old_start,new_start,old_end,new_end,start_shift_ms,end_shift_ms,model,label_version
0,c2da076a65ceaf6a699a:n00000,c2da076a65ceaf6a699a,timing_change,0.394739,0.374437,0.684989,0.708476,-20.3,23.5,"RandomForestRegressor(n_estimators=150, min_sa...",proposed_v2/train_targets_v2.csv (provisional)
1,c2da076a65ceaf6a699a:n00001,c2da076a65ceaf6a699a,timing_change,0.777868,0.712780,0.986848,1.047344,-65.1,60.5,"RandomForestRegressor(n_estimators=150, min_sa...",proposed_v2/train_targets_v2.csv (provisional)
2,c2da076a65ceaf6a699a:n00002,c2da076a65ceaf6a699a,timing_change,1.021678,0.984461,1.265488,1.346917,-37.2,81.4,"RandomForestRegressor(n_estimators=150, min_sa...",proposed_v2/train_targets_v2.csv (provisional)
3,c2da076a65ceaf6a699a:n00003,c2da076a65ceaf6a699a,timing_change,1.416417,1.331848,1.625397,1.711233,-84.6,85.8,"RandomForestRegressor(n_estimators=150, min_sa...",proposed_v2/train_targets_v2.csv (provisional)
4,c2da076a65ceaf6a699a:n00004,c2da076a65ceaf6a699a,timing_change,1.718277,1.646517,1.915646,1.978627,-71.8,63.0,"RandomForestRegressor(n_estimators=150, min_sa...",proposed_v2/train_targets_v2.csv (provisional)


## Step 4b: choose the gate fairly (nested cross-validation)

A **gate** skips small predicted shifts: if |start shift| < start gate, or |end shift| < end gate, that boundary stays where Basic Pitch put it.

The 30/80 ms gate in Step 3 was picked by looking at the same scores we report, which is slightly optimistic. Here, each held-out fold *k* gets its own gate, chosen **without ever looking at fold k**:
1. On the 4 training folds only, make inner out-of-fold RF predictions (train on 3, predict the 4th).
2. Try each start gate and keep the one with the best onset F1 on those 4 folds. Then try each end gate and keep the best offset F1.
3. Apply that gate to fold k's real predictions from cell 7.

This takes about 3 minutes (40 extra forests and roughly 100 scoring runs).

In [8]:
# Step 4b: choose the gate fairly (nested), then score and export the gated random forest

make_rf = lambda: RandomForestRegressor(n_estimators=150, min_samples_leaf=20, max_features=0.6, n_jobs=-1, random_state=0)
START_GATES = [0.00, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08]   # seconds
END_GATES   = [0.00, 0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.14, 0.16, 0.18, 0.20, 1.0]   # 1.0 s = never move ends
rec_fold = full_manifest['sample_id'].map(folds)              # fold of every training recording
lab = usable.to_numpy()

def f1_on(fold_set, shift, gate_start, gate_end):
    """End-to-end onset/offset F1, scored only on the recordings of the songs in fold_set."""
    keep_rec = full_manifest[rec_fold.isin(fold_set)]
    corrected = apply_shifts(shift, gate_start, gate_end)
    corrected = corrected[corrected['recording_id'].isin(keep_rec['recording_id'])]
    refs = full_references[full_references['recording_id'].isin(keep_rec['recording_id'])]
    s = cu.evaluate_end_to_end(corrected, refs, keep_rec)['summary']
    return s['song_macro_f1'], s['song_macro_offset_f1']

chosen = {}
gated_shift = shift_rf.copy()
for k in sorted(fold_all.unique()):
    others = [j for j in sorted(fold_all.unique()) if j != k]       # the 4 training folds for outer fold k

    # 1. Inner out-of-fold predictions on the 4 training folds only (fold k is never used).
    inner = np.zeros((len(X), 2))
    for j in others:
        test  = (fold_all == j).to_numpy()
        train = fold_all.isin([o for o in others if o != j]).to_numpy() & lab
        for col, y in enumerate([y_start, y_end]):
            inner[test, col] = make_rf().fit(X[train], y[train]).predict(X[test])

    # 2. Pick the start gate by onset F1, then the end gate by offset F1, on those 4 folds.
    gs = max(START_GATES, key=lambda g: f1_on(others, inner, g, 0.0)[0])
    ge = max(END_GATES,   key=lambda g: f1_on(others, inner, gs, g)[1])
    chosen[int(k)] = (gs, ge)

    # 3. Apply that gate to fold k's real out-of-fold predictions from cell 7.
    rows = (fold_all == k).to_numpy()
    gated_shift[rows & (np.abs(shift_rf[:, 0]) < gs), 0] = 0
    gated_shift[rows & (np.abs(shift_rf[:, 1]) < ge), 1] = 0
    print(f'fold {k}: start gate {gs*1000:.0f} ms, end gate {ge*1000:.0f} ms')

results.loc['RF, nested gate'] = pd.Series(score('RF, nested gate', apply_shifts(gated_shift))).drop('model')
display(results)

corrected_rf_gate, history_rf_gate = export_version(
    'rf_v2_nestedgate', gated_shift,
    f'RandomForestRegressor(n_estimators=150, min_samples_leaf=20, max_features=0.6), out-of-fold by song; '
    f'gates chosen per fold by inner song-grouped CV (start/end s): {chosen}')

fold 0: start gate 60 ms, end gate 180 ms
fold 1: start gate 50 ms, end gate 180 ms
fold 2: start gate 30 ms, end gate 1000 ms
fold 3: start gate 40 ms, end gate 200 ms
fold 4: start gate 50 ms, end gate 160 ms


,onset_mae_ms,onset_within_50ms,offset_mae_ms,offset_within_tol,onset_F1,offset_F1
model,,,,,,
Basic Pitch unchanged,68.8,0.493,91.8,0.721,0.2628,0.1539
Decision tree,56.4,0.531,99.5,0.651,0.2899,0.1573
Random forest,54.9,0.531,93.0,0.680,0.2882,0.1636
"RF, gated 30/80 ms *",54.9,0.539,92.5,0.705,0.2923,0.1742
"RF, nested gate",56.9,0.538,91.8,0.720,0.2915,0.1748


Already saved, unchanged: le_xin_timing_rf_v2_nestedgate.csv
Already saved, unchanged: le_xin_timing_rf_v2_nestedgate_actions.csv
  7623 notes kept, 0 deleted, 3867 timing changes (3823 starts, 53 ends moved)


## Step 5a: audio-only features near each boundary

The note features can't tell *where* a particular note really starts; only the audio can. For every note we look ±100 ms around its start and end in two loudness curves:
- **onset strength (spectral flux):** spikes when a new sound begins;
- **RMS loudness in dB:** its frame-to-frame rise (attack) and drop (release).

From these we record *where* the strongest attack, rise or drop is (relative to Basic Pitch's boundary, in seconds) and *how strong* it is.

These are **audio-only** features computed from the training WAVs. No TextGrid and no reference data are used, so nothing leaks the answer. Results are cached in `le_xin_cache/` so later runs skip the ~30 s computation.

In [9]:
# Step 5a: audio-only features around each note's start and end (no TextGrid, no reference data)
import librosa

ROOT = (HERE / '..' / '..').resolve()               # GTSinger_100_unique_sample
SR, HOP, WIN = 22050, 256, 0.10                     # 22.05 kHz audio, ~11.6 ms frames, look +-100 ms
CACHE = HERE / 'le_xin_cache' / 'audio_features_train_v1.csv'

def curves(path):
    """Two loudness curves for one recording, one value per ~11.6 ms frame."""
    y, _ = librosa.load(path, sr=SR, mono=True)
    flux = librosa.onset.onset_strength(y=y, sr=SR, hop_length=HOP)          # 'new sound starts here'
    flux = flux / (np.percentile(flux, 99) + 1e-9)                           # scale per recording
    db = librosa.amplitude_to_db(librosa.feature.rms(y=y, hop_length=HOP)[0], ref=np.max)
    rise = np.diff(db, prepend=db[0])                                        # dB change per frame
    times = librosa.frames_to_time(np.arange(len(flux)), sr=SR, hop_length=HOP)
    return times, flux, db, rise

def near(times, values, t, pick):
    """Within t +- WIN: (time of the strongest point minus t, its value). pick = np.argmax or np.argmin."""
    w = np.flatnonzero(np.abs(times - t) <= WIN)
    if len(w) == 0:
        return 0.0, 0.0
    i = w[pick(values[w])]
    return times[i] - t, values[i]

def audio_features(man=None, nts=None, split='train'):
    man = manifest if man is None else man
    nts = notes if nts is None else nts
    assert split in ('train', 'val')                                              # test audio stays locked
    rows = []
    for rec in man.itertuples():
        assert rec.split == split and rec.audio_path.startswith(f'data/{split}/')
        times, flux, db, rise = curves(ROOT / rec.audio_path)
        for n in nts[nts['recording_id'] == rec.recording_id].itertuples():
            s_flux_dt, s_flux = near(times, flux, n.start, np.argmax)    # strongest attack near the start
            s_rise_dt, s_rise = near(times, rise, n.start, np.argmax)    # biggest loudness jump near the start
            e_drop_dt, e_drop = near(times, rise, n.end,   np.argmin)    # biggest loudness drop near the end
            e_flux_dt, e_flux = near(times, flux, n.end,   np.argmax)    # next attack near the end
            frame = lambda t: min(int(round(t * SR / HOP)), len(db) - 1)
            rows.append(dict(note_id=n.note_id,
                             s_flux_dt=s_flux_dt, s_flux=s_flux, s_rise_dt=s_rise_dt, s_rise=s_rise,
                             e_drop_dt=e_drop_dt, e_drop=e_drop, e_flux_dt=e_flux_dt, e_flux=e_flux,
                             db_at_start=db[frame(n.start)], db_at_end=db[frame(n.end)]))
    return pd.DataFrame(rows)

if CACHE.exists():
    audio = pd.read_csv(CACHE, dtype={'note_id':str})
    print('Loaded cached audio features:', CACHE.name)
else:
    audio = audio_features()                         # ~30 s for 534 recordings
    CACHE.parent.mkdir(exist_ok=True)
    audio.to_csv(CACHE, index=False)
    print('Computed and cached audio features:', CACHE.name)
audio = joined[['note_id']].merge(audio, on='note_id', how='left', validate='one_to_one')
assert audio.notna().all().all() and len(audio) == len(joined)
AUDIO_COLS = [c for c in audio.columns if c != 'note_id']

# Sanity check on labelled notes: does the strongest nearby attack point the same way as the label?
lab_true = y_start[usable] * 1000
print(f"\nLabelled notes: true start shift median {lab_true.median():.0f} ms")
for col in ['s_flux_dt', 's_rise_dt']:
    guess = audio.loc[usable, col] * 1000
    print(f"  {col}: median {guess.median():.0f} ms | correlation with true shift {np.corrcoef(guess, lab_true)[0,1]:.2f}"
          f" | snapping start there lands within 50 ms for {np.mean(np.abs(guess - lab_true) <= 50):.1%}"
          f" (unchanged: {np.mean(np.abs(lab_true) <= 50):.1%})")
display(audio.loc[usable].head())

Loaded cached audio features: audio_features_train_v1.csv

Labelled notes: true start shift median -37 ms
  s_flux_dt: median 16 ms | correlation with true shift 0.29 | snapping start there lands within 50 ms for 36.0% (unchanged: 49.3%)
  s_rise_dt: median -14 ms | correlation with true shift 0.27 | snapping start there lands within 50 ms for 47.4% (unchanged: 49.3%)


,note_id,s_flux_dt,s_flux,s_rise_dt,s_rise,e_drop_dt,e_drop,e_flux_dt,e_flux,db_at_start,db_at_end
0,c2da076a65ceaf6a699a:n00000,-0.02322,1.405158,0.01161,4.203056,0.01161,-2.950237,0.03483,0.942590,-18.168282,-9.725281
1,c2da076a65ceaf6a699a:n00001,-0.05805,0.942590,-0.04644,3.220436,0.02322,-0.854038,0.08127,1.009560,-4.658527,-8.739330
2,c2da076a65ceaf6a699a:n00002,0.04644,1.009560,0.05805,0.921631,0.03483,-4.278153,0.03483,0.175275,-10.151955,-8.097244
3,c2da076a65ceaf6a699a:n00003,0.00000,0.280074,-0.06966,6.371998,-0.01161,-0.191746,0.05805,0.312639,-6.580017,-6.780251
4,c2da076a65ceaf6a699a:n00004,-0.03483,0.312639,-0.02322,0.413292,0.01161,-4.108349,0.02322,0.319825,-5.120903,-12.227413


## Step 5b: rule vs decision tree vs random forest, with audio features

Same out-of-fold-by-song setup as cell 7, now with 17 features (7 note + 10 audio):
1. **Rule:** move each start to the biggest nearby loudness rise and each end to the biggest drop. No learning.
2. **Decision tree** and **random forest** on the 17 features.
3. **Random forest + nested gate:** gates chosen fairly per fold, as in Step 4b (about 3 minutes).
4. **Cap ends:** also end each note no later than the next note's start, to remove overlaps.

The fair model (RF + audio, nested gate) is exported with its action history.

In [10]:
# Step 5b: rule vs decision tree vs random forest, now with the audio features

XA = pd.concat([X, audio[AUDIO_COLS]], axis=1)       # 7 note features + 10 audio features
make_dt = lambda: DecisionTreeRegressor(min_samples_leaf=40, random_state=0)

def oof_on(Xm, make_model, fold_set=None):
    """Out-of-fold shifts by song, using feature matrix Xm. With fold_set, only those folds are used
    (train on the others in fold_set, predict each one); rows outside fold_set stay 0."""
    fold_set = sorted(fold_all.unique()) if fold_set is None else fold_set
    pred = np.zeros((len(Xm), 2))
    for k in fold_set:
        test  = (fold_all == k).to_numpy()
        train = fold_all.isin([o for o in fold_set if o != k]).to_numpy() & lab
        for col, y in enumerate([y_start, y_end]):
            pred[test, col] = make_model().fit(Xm[train], y[train]).predict(Xm[test])
    return pred

def nested_gate(Xm, make_model, outer_shift):
    """Step 4b's procedure as a function: per held-out fold, choose gates on the other 4 folds only."""
    gated, picked = outer_shift.copy(), {}
    for k in sorted(fold_all.unique()):
        others = [j for j in sorted(fold_all.unique()) if j != k]
        inner = oof_on(Xm, make_model, others)
        gs = max(START_GATES, key=lambda g: f1_on(others, inner, g, 0.0)[0])
        ge = max(END_GATES,   key=lambda g: f1_on(others, inner, gs, g)[1])
        picked[int(k)] = (gs, ge)
        rows = (fold_all == k).to_numpy()
        gated[rows & (np.abs(outer_shift[:, 0]) < gs), 0] = 0
        gated[rows & (np.abs(outer_shift[:, 1]) < ge), 1] = 0
    return gated, picked

def cap_overlaps(corrected):
    """Monophonic singing: end each note no later than the next note's start (same recording)."""
    out = corrected.sort_values(['recording_id', 'start']).copy()
    nxt = out.groupby('recording_id')['start'].shift(-1)
    cap = nxt.notna() & (out['end'] > nxt) & (nxt >= out['start'] + 0.01)
    out.loc[cap, 'end'] = nxt[cap]
    return out.loc[corrected.index]

# 1. Rule (no learning): move the start to the biggest loudness rise, the end to the biggest drop.
rule_shift = audio[['s_rise_dt', 'e_drop_dt']].to_numpy()

# 2. Learned models with audio features, out-of-fold by song.
shift_dt_a = oof_on(XA, make_dt)
shift_rf_a = oof_on(XA, make_rf)

# 3. Fair gate for the audio random forest (about 3 minutes, like Step 4b).
gated_rf_a, chosen_a = nested_gate(XA, make_rf, shift_rf_a)
for k, (gs, ge) in chosen_a.items():
    print(f'fold {k}: start gate {gs*1000:.0f} ms, end gate {ge*1000:.0f} ms')

def add(name, corrected):
    results.loc[name] = pd.Series(score(name, corrected)).drop('model')

add('Rule: snap to loudness (audio)',      apply_shifts(rule_shift))
add('DT + audio',                          apply_shifts(shift_dt_a))
add('RF + audio',                          apply_shifts(shift_rf_a))
add('RF + audio, nested gate',             apply_shifts(gated_rf_a))
add('RF + audio, nested gate, cap ends', cap_overlaps(apply_shifts(gated_rf_a)))
display(results)

# 4. Export the fair audio model (nested gate, no capping) with its action history.
corrected_rf_audio, history_rf_audio = export_version(
    'rf_audio_v2_nestedgate', gated_rf_a,
    'RandomForestRegressor(n_estimators=150, min_samples_leaf=20, max_features=0.6) on 7 note + 10 audio-only '
    f'features (librosa onset strength + RMS, +-100 ms, no TextGrid), out-of-fold by song; nested gates (s): {chosen_a}')

fold 0: start gate 30 ms, end gate 100 ms
fold 1: start gate 40 ms, end gate 120 ms
fold 2: start gate 40 ms, end gate 100 ms
fold 3: start gate 30 ms, end gate 100 ms
fold 4: start gate 30 ms, end gate 120 ms


,onset_mae_ms,onset_within_50ms,offset_mae_ms,offset_within_tol,onset_F1,offset_F1
model,,,,,,
Basic Pitch unchanged,68.8,0.493,91.8,0.721,0.2628,0.1539
Decision tree,56.4,0.531,99.5,0.651,0.2899,0.1573
Random forest,54.9,0.531,93.0,0.680,0.2882,0.1636
"RF, gated 30/80 ms *",54.9,0.539,92.5,0.705,0.2923,0.1742
"RF, nested gate",56.9,0.538,91.8,0.720,0.2915,0.1748
Rule: snap to loudness (audio),65.5,0.474,91.6,0.707,0.2634,0.1540
DT + audio,52.6,0.590,97.1,0.669,0.3260,0.1822
RF + audio,49.3,0.616,88.6,0.699,0.3407,0.1977
"RF + audio, nested gate",49.8,0.626,88.4,0.729,0.3437,0.2083


Already saved, unchanged: le_xin_timing_rf_audio_v2_nestedgate.csv
Already saved, unchanged: le_xin_timing_rf_audio_v2_nestedgate_actions.csv
  7623 notes kept, 0 deleted, 5186 timing changes (4690 starts, 1057 ends moved)


## Step 5c: pitch-change features (audio-only)

Singers often glide from one note to the next (legato) without a loudness jump, so Step 5a's loudness curves miss those starts. A pitch tracker (`librosa.pyin`) tells us, every ~11.6 ms, *which pitch* is being sung and whether the voice is on at all. Around each start and end (±100 ms) we record:
- where the voice **switches on/off** (silence → voice, voice → silence);
- where the **pitch changes fastest**, and by how much;
- the **voicing probability** at the boundary;
- how far the sung pitch in the 50 ms **before/after** the boundary is from the note's pitch. For example, if the singer is already on this note's pitch *before* Basic Pitch's start, the start is probably late.

That gives 14 more audio-only features (no TextGrid, no reference data), cached in `le_xin_cache/` (~5 minutes the first time). The random forest is retrained with all 31 features, using the same nested gate as Step 5b (another ~3–4 minutes), and the fair model is exported.

In [11]:
# Step 5c: audio-only PITCH features near each boundary (pyin pitch tracker; no TextGrid, no reference data)

PCACHE = HERE / 'le_xin_cache' / 'pitch_features_train_v1.csv'
FAR = 12.0          # 'no pitch found' stand-in: one octave away

def pitch_track(path):
    """Pitch (MIDI, NaN when unvoiced) and voicing probability, one value per ~11.6 ms frame."""
    y, _ = librosa.load(path, sr=SR, mono=True)
    f0, voiced, vprob = librosa.pyin(y, fmin=65, fmax=1100, sr=SR, hop_length=HOP)
    midi = np.where(voiced, librosa.hz_to_midi(np.nan_to_num(f0, nan=1.0)), np.nan)
    jump = np.abs(np.gradient(np.nan_to_num(midi, nan=0.0)))            # semitones per frame
    jump[np.isnan(midi)] = 0.0                                          # ignore jumps into/out of silence
    times = librosa.frames_to_time(np.arange(len(midi)), sr=SR, hop_length=HOP)
    return times, midi, voiced.astype(bool), vprob, jump

def transition_dt(times, voiced, t, to_voiced):
    """Time from t to the nearest silence->voice (or voice->silence) switch within +-WIN; (dt, found)."""
    switch = np.flatnonzero(voiced[1:] != voiced[:-1]) + 1
    switch = switch[voiced[switch] == to_voiced]
    dts = times[switch] - t
    dts = dts[np.abs(dts) <= WIN]
    return (dts[np.argmin(np.abs(dts))], 1.0) if len(dts) else (0.0, 0.0)

def pitch_gap(times, midi, a, b, note_pitch):
    """How far (semitones) the sung pitch between times a and b is from the note's pitch."""
    seg = midi[(times >= a) & (times < b)]
    seg = seg[~np.isnan(seg)]
    return min(abs(np.median(seg) - note_pitch), FAR) if len(seg) else FAR

def pitch_features(man=None, nts=None, split='train'):
    man = manifest if man is None else man
    nts = notes if nts is None else nts
    assert split in ('train', 'val')                                              # test audio stays locked
    rows = []
    for rec in man.itertuples():
        assert rec.split == split and rec.audio_path.startswith(f'data/{split}/')
        times, midi, voiced, vprob, jump = pitch_track(ROOT / rec.audio_path)
        frame = lambda t: min(int(round(t * SR / HOP)), len(times) - 1)
        for n in nts[nts['recording_id'] == rec.recording_id].itertuples():
            von_dt, von_found   = transition_dt(times, voiced, n.start, True)    # voice switches on near start
            voff_dt, voff_found = transition_dt(times, voiced, n.end,   False)   # voice switches off near end
            sj_dt, sj = near(times, jump, n.start, np.argmax)                    # biggest pitch change near start
            ej_dt, ej = near(times, jump, n.end,   np.argmax)                    # biggest pitch change near end
            rows.append(dict(note_id=n.note_id,
                             p_von_dt=von_dt, p_von_found=von_found, p_voff_dt=voff_dt, p_voff_found=voff_found,
                             p_sjump_dt=sj_dt, p_sjump=sj, p_ejump_dt=ej_dt, p_ejump=ej,
                             p_vprob_start=vprob[frame(n.start)], p_vprob_end=vprob[frame(n.end)],
                             # already on this note's pitch before the start? -> start is probably late
                             p_gap_before_start=pitch_gap(times, midi, n.start - 0.05, n.start, n.pitch),
                             p_gap_after_start =pitch_gap(times, midi, n.start, n.start + 0.05, n.pitch),
                             # still on this note's pitch after the end? -> end is probably early
                             p_gap_before_end  =pitch_gap(times, midi, n.end - 0.05, n.end, n.pitch),
                             p_gap_after_end   =pitch_gap(times, midi, n.end, n.end + 0.05, n.pitch)))
    return pd.DataFrame(rows)

if PCACHE.exists():
    pitchf = pd.read_csv(PCACHE, dtype={'note_id':str})
    print('Loaded cached pitch features:', PCACHE.name)
else:
    pitchf = pitch_features()                        # ~5 minutes for 534 recordings
    PCACHE.parent.mkdir(exist_ok=True)
    pitchf.to_csv(PCACHE, index=False)
    print('Computed and cached pitch features:', PCACHE.name)
pitchf = joined[['note_id']].merge(pitchf, on='note_id', how='left', validate='one_to_one')
assert pitchf.notna().all().all() and len(pitchf) == len(joined)
PITCH_COLS = [c for c in pitchf.columns if c != 'note_id']

# Sanity check on labelled notes (same idea as Step 5a)
for col in ['p_von_dt', 'p_sjump_dt']:
    guess = pitchf.loc[usable, col] * 1000
    print(f"  {col}: correlation with true start shift {np.corrcoef(guess, lab_true)[0,1]:.2f}"
          f" | snapping start there lands within 50 ms for {np.mean(np.abs(guess - lab_true) <= 50):.1%}")
late = pitchf.loc[usable, 'p_gap_before_start'] < 0.5
print(f"  notes already on their pitch in the 50 ms BEFORE the start: {late.mean():.1%}; "
      f"their median true start shift {lab_true[late].median():.0f} ms vs {lab_true[~late].median():.0f} ms for the rest")

# Random forest with note + loudness + pitch features (same setup as Step 5b)
XAP = pd.concat([X, audio[AUDIO_COLS], pitchf[PITCH_COLS]], axis=1)     # 7 + 10 + 14 = 31 features
shift_rf_ap = oof_on(XAP, make_rf)
gated_rf_ap, chosen_ap = nested_gate(XAP, make_rf, shift_rf_ap)          # about 3-4 minutes
for k, (gs, ge) in chosen_ap.items():
    print(f'fold {k}: start gate {gs*1000:.0f} ms, end gate {ge*1000:.0f} ms')
add('RF + audio + pitch',               apply_shifts(shift_rf_ap))
add('RF + audio + pitch, nested gate',  apply_shifts(gated_rf_ap))
display(results)

# Export the fair model (nested gate) with its action history.
corrected_rf_ap, history_rf_ap = export_version(
    'rf_audio_pitch_v2_nestedgate', gated_rf_ap,
    'RandomForestRegressor(n_estimators=150, min_samples_leaf=20, max_features=0.6) on 7 note + 10 loudness + 14 pitch '
    f'audio-only features (librosa onset strength, RMS, pyin; +-100 ms; no TextGrid), out-of-fold by song; nested gates (s): {chosen_ap}')

Loaded cached pitch features: pitch_features_train_v1.csv
  p_von_dt: correlation with true start shift 0.23 | snapping start there lands within 50 ms for 53.8%
  p_sjump_dt: correlation with true start shift 0.33 | snapping start there lands within 50 ms for 61.4%
  notes already on their pitch in the 50 ms BEFORE the start: 19.6%; their median true start shift -68 ms vs -28 ms for the rest
fold 0: start gate 30 ms, end gate 60 ms
fold 1: start gate 40 ms, end gate 80 ms
fold 2: start gate 30 ms, end gate 100 ms
fold 3: start gate 40 ms, end gate 60 ms
fold 4: start gate 40 ms, end gate 80 ms


,onset_mae_ms,onset_within_50ms,offset_mae_ms,offset_within_tol,onset_F1,offset_F1
model,,,,,,
Basic Pitch unchanged,68.8,0.493,91.8,0.721,0.2628,0.1539
Decision tree,56.4,0.531,99.5,0.651,0.2899,0.1573
Random forest,54.9,0.531,93.0,0.680,0.2882,0.1636
"RF, gated 30/80 ms *",54.9,0.539,92.5,0.705,0.2923,0.1742
"RF, nested gate",56.9,0.538,91.8,0.720,0.2915,0.1748
Rule: snap to loudness (audio),65.5,0.474,91.6,0.707,0.2634,0.1540
DT + audio,52.6,0.590,97.1,0.669,0.3260,0.1822
RF + audio,49.3,0.616,88.6,0.699,0.3407,0.1977
"RF + audio, nested gate",49.8,0.626,88.4,0.729,0.3437,0.2083


Already saved, unchanged: le_xin_timing_rf_audio_pitch_v2_nestedgate.csv
Already saved, unchanged: le_xin_timing_rf_audio_pitch_v2_nestedgate_actions.csv
  7623 notes kept, 0 deleted, 5738 timing changes (4835 starts, 2392 ends moved)


## Step 6: validate on the validation songs (`data/val`)

So far every score came from training songs, held out one fold at a time. Now we check the models on the **20 validation songs** (160 recordings), which no model has trained on.
- Validation has **no correction labels**, so there are no task metrics, only end-to-end onset/offset F1.
- Validation **reference notes** are parsed from `data/val/*.json` by the project's own parser (`bp_experiment.references`), the same one that built `references_train.csv`. They are only used for scoring.
- Each **final model** is trained on *all* labelled training notes. Its gate is chosen on the training songs' out-of-fold scores, so validation plays no part in any choice.
- The audio features for the val notes are computed the same way and cached in `le_xin_cache/` (~2 minutes the first time).
- `data/test` stays locked.

The README says validation was already used to choose Basic Pitch's settings, so this counts as continued development, not untouched final evidence.

In [12]:
# Step 6: validate on the 20 validation songs (data/val). No model has trained on them and they have no labels.
sys.path.insert(0, str((HERE / '..').resolve()))
import bp_experiment as bx                          # the project's own reference parser (built references_train.csv)

all_manifest = pd.read_csv(DATA / 'recording_manifest.csv')
val_manifest = all_manifest[all_manifest['split'] == 'val'].reset_index(drop=True)
val_notes = pd.read_csv(DATA / 'val_features.csv', dtype={'note_id':str, 'recording_id':str, 'sample_id':str})
assert val_notes['split'].eq('val').all() and val_manifest['split'].eq('val').all()   # never test
cu.validate_notes(val_notes, val_manifest)

# 1. Validation reference notes (evaluator only, never model input), parsed exactly like the training ones.
val_events, _ = bx.references(val_manifest.to_dict('records'))
val_references = pd.DataFrame([dict(note_id=f'{rid}:ref{i:05d}', recording_id=rid, start=e[0], end=e[1], pitch=e[2])
                               for rid, events in val_events.items() for i, e in enumerate(events)])
print(f'Validation: {val_manifest.sample_id.nunique()} songs, {len(val_manifest)} recordings, '
      f'{len(val_notes)} Basic Pitch notes, {len(val_references)} reference notes')

# 2. The same 31 features for the validation notes (audio features cached like the training ones).
def cached(path, compute):
    if path.exists():
        return pd.read_csv(path, dtype={'note_id':str})
    table = compute()
    table.to_csv(path, index=False)
    return table
audio_val = cached(HERE / 'le_xin_cache' / 'audio_features_val_v1.csv',
                   lambda: audio_features(val_manifest, val_notes, 'val'))      # ~10 s
pitch_val = cached(HERE / 'le_xin_cache' / 'pitch_features_val_v1.csv',
                   lambda: pitch_features(val_manifest, val_notes, 'val'))      # ~1.5 minutes
audio_val = val_notes[['note_id']].merge(audio_val, on='note_id', how='left', validate='one_to_one')
pitch_val = val_notes[['note_id']].merge(pitch_val, on='note_id', how='left', validate='one_to_one')
assert audio_val.notna().all().all() and pitch_val.notna().all().all()
X_val   = val_notes[X.columns].astype(float)
XA_val  = pd.concat([X_val, audio_val[AUDIO_COLS]], axis=1)
XAP_val = pd.concat([XA_val, pitch_val[PITCH_COLS]], axis=1)

# 3. Final model = trained on ALL labelled training notes. Its gate is chosen on the training songs'
#    out-of-fold scores (all 5 folds), so validation plays no part in any choice.
all_folds = sorted(fold_all.unique())
val_end = val_notes['recording_id'].map(val_manifest.set_index('recording_id')['duration_seconds_if_validated']).to_numpy()

def final_shift(X_train, X_valid, oof_shift):
    gs = max(START_GATES, key=lambda g: f1_on(all_folds, oof_shift, g, 0.0)[0])
    ge = max(END_GATES,   key=lambda g: f1_on(all_folds, oof_shift, gs, g)[1])
    shift = np.column_stack([make_rf().fit(X_train[lab], y[lab]).predict(X_valid) for y in (y_start, y_end)])
    return shift, gs, ge

def score_val(corrected):
    out = cu.evaluate_end_to_end(corrected, val_references, val_manifest)
    assert out['summary']['recordings'] == len(val_manifest)
    return out

unchanged_val = score_val(val_notes[cu.NOTE_COLUMNS])
base_song_f1 = {s['sample_id']: s['f1'] for s in unchanged_val['songs']}
rows = [dict(model='Basic Pitch unchanged', val_onset_F1=round(unchanged_val['summary']['song_macro_f1'], 4),
             val_offset_F1=round(unchanged_val['summary']['song_macro_offset_f1'], 4))]
val_corrected = {}
for name, X_train, X_valid, oof_shift, train_row in [
        ('RF, note features',  X,   X_val,   shift_rf,    'RF, nested gate'),
        ('RF + audio',         XA,  XA_val,  shift_rf_a,  'RF + audio, nested gate'),
        ('RF + audio + pitch', XAP, XAP_val, shift_rf_ap, 'RF + audio + pitch, nested gate')]:
    shift, gs, ge = final_shift(X_train, X_valid, oof_shift)
    val_corrected[name] = apply_shifts(shift, gs, ge, base=val_notes, end_limit=val_end)
    out = score_val(val_corrected[name])
    better = sum(s['f1'] > base_song_f1[s['sample_id']] for s in out['songs'])
    rows.append(dict(model=name, val_onset_F1=round(out['summary']['song_macro_f1'], 4),
                     val_offset_F1=round(out['summary']['song_macro_offset_f1'], 4),
                     songs_improved=f"{better}/{len(out['songs'])}",
                     start_gate_ms=round(gs * 1000), end_gate_ms=round(ge * 1000),
                     train_OOF_onset_F1=results.loc[train_row, 'onset_F1'],
                     train_OOF_offset_F1=results.loc[train_row, 'offset_F1']))
val_results = pd.DataFrame(rows).set_index('model')
display(val_results)
print('Published validation baseline for unchanged Basic Pitch: 0.2726 / 0.1602')

# 4. Export the final model's corrected validation notes + action history (write once, like Step 4).
final_val = val_corrected['RF + audio + pitch']
old_v, new_v = val_notes.set_index('note_id'), final_val.set_index('note_id')
moved = (new_v['start'] - old_v['start']).abs().gt(1e-9) | (new_v['end'] - old_v['end']).abs().gt(1e-9)
val_history = pd.DataFrame({'note_id': new_v.index[moved], 'recording_id': new_v.loc[moved, 'recording_id'].to_numpy(),
                            'action': 'timing_change',
                            'old_start': old_v.loc[moved, 'start'].to_numpy(), 'new_start': new_v.loc[moved, 'start'].to_numpy(),
                            'old_end': old_v.loc[moved, 'end'].to_numpy(),     'new_end': new_v.loc[moved, 'end'].to_numpy()})
val_history['model'] = 'RF + audio + pitch (31 audio-only/note features), trained on all labelled training notes'
val_history['label_version'] = LABEL_VERSION
for table, path, check in [(final_val[cu.NOTE_COLUMNS], EXPORTS / 'le_xin_timing_val_rf_audio_pitch.csv', True),
                           (val_history, EXPORTS / 'le_xin_timing_val_rf_audio_pitch_actions.csv', False)]:
    if path.exists():
        saved = pd.read_csv(path, dtype={'note_id':str, 'recording_id':str})
        pd.testing.assert_frame_equal(saved, table.reset_index(drop=True), check_dtype=False)
        print('Already saved, unchanged:', path.name)
    elif check:
        cu.export_corrections(table, val_manifest, path); print('Saved:', path.name)
    else:
        with path.open('x', encoding='utf-8', newline='') as handle:
            table.to_csv(handle, index=False)
        print('Saved:', path.name)

Validation: 20 songs, 160 recordings, 2329 Basic Pitch notes, 2501 reference notes


,val_onset_F1,val_offset_F1,songs_improved,start_gate_ms,end_gate_ms,train_OOF_onset_F1,train_OOF_offset_F1
model,,,,,,,
Basic Pitch unchanged,0.2726,0.1602,NaN,NaN,NaN,NaN,NaN
"RF, note features",0.3115,0.1866,13/20,40.0,180.0,0.2915,0.1748
RF + audio,0.3657,0.2279,20/20,40.0,80.0,0.3437,0.2083
RF + audio + pitch,0.3755,0.2360,20/20,40.0,80.0,0.3541,0.2176


Published validation baseline for unchanged Basic Pitch: 0.2726 / 0.1602
Saved: le_xin_timing_val_rf_audio_pitch.csv
Saved: le_xin_timing_val_rf_audio_pitch_actions.csv


## Full end-to-end scoring (when the evaluator tables are ready)

Use `cu.evaluate_end_to_end(corrected, full_references, evaluation_manifest)` for both the unchanged baseline and future corrected notes. All three tables must describe the same declared train/val scope. `evaluation_manifest` has `recording_id, sample_id, split` and must include recordings with zero predictions. `full_references` has `note_id, recording_id, start, end, pitch`; reference IDs are independent of prediction IDs.

Do **not** build this manifest from a filtered prediction table, or filter references with `usable`, task matches, or ambiguity flags. The manifest loaded above is the complete training recording manifest, including zero-prediction recordings. Reference tables are evaluator-only, never feature inputs.

The shared evaluator reuses `bp_tuning_core.score_notes` and `aggregate`: 50 ms onset, 50-cent pitch, optional offsets (20% or 50 ms), song-pooled macro scores and pooled micro F1. The next cell runs it on every training recording using evaluator-only references. It never filters on task-label availability. No validation or test labels are opened.


In [13]:
full_manifest = pd.read_csv(DATA / 'training_recording_manifest.csv')
full_references = pd.read_csv(DATA / 'references_train.csv')
end_to_end = cu.evaluate_end_to_end(unchanged, full_references, full_manifest)
print('All-training no-correction baseline:', end_to_end['summary'])
assert end_to_end['summary']['recordings'] == len(full_manifest)


All-training no-correction baseline: {'song_macro_f1': 0.26275846642410683, 'song_macro_offset_f1': 0.15386648702425776, 'song_macro_precision': 0.27330092213557544, 'song_macro_recall': 0.2582440522033118, 'micro_f1': 0.26521100683138604, 'songs': 60, 'recordings': 534}
